# Vežba 6: Evaluacija i prag odlučivanja

Matrica konfuzije, preciznost i odziv, ROC i PR kriva, retka klasa i izbor praga prema ceni greške

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba6.html](https://isatvss.surge.sh/vezba6.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Matrica konfuzije

### Zadatak 1. Model sa vežbe 5

Pripremiti podatke i istrenirati logističku regresiju kao na vežbi 5, a zatim izračunati verovatnoće i procene sa pragom 0,5 za test skup.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode = nezgode.drop_duplicates(subset="id_nezgode").reset_index(drop=True)
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)
nezgode["poginuli"] = (nezgode["posledica"] == "Sa poginulim").astype(int)
vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")
nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month

numericke = ["sat", "dan_u_nedelji", "mesec", "lon", "lat"]
kategoricke = ["policijska_uprava", "vrsta", "opis"]
X = nezgode[numericke + kategoricke]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, nezgode["teska"], test_size=0.2, stratify=nezgode["teska"], random_state=42
)


def priprema():
    numericki_deo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    return ColumnTransformer(
        [
            ("num", numericki_deo, numericke),
            ("kat", OneHotEncoder(handle_unknown="ignore"), kategoricke),
        ]
    )


model = Pipeline(
    [("priprema", priprema()), ("model", LogisticRegression(max_iter=1000))]
)
model.fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]
procena = (p_te >= 0.5).astype(int)
print(f"Test nezgoda: {len(y_te)}, od toga teških: {int(y_te.sum())}")

### Zadatak 2. Matrica konfuzije

Napraviti matricu konfuzije na dva načina, tabelom učestanosti i funkcijom `confusion_matrix`, izdvojiti četiri broja i nacrtati matricu.

Od 2.693 teške nezgode u testu model je pronašao 1789, a propustio 904. Uz to je 773 lake nezgode proglasio teškim. Ovi brojevi kažu više od tačnosti 0,747 iz vežbe 5, jer pokazuju koje greške model pravi.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

print(pd.crosstab(y_te.values, procena, rownames=["stvarno"], colnames=["procena"]))

tn, fp, fn, tp = confusion_matrix(y_te, procena).ravel()
print(f"\nTN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

ConfusionMatrixDisplay.from_predictions(
    y_te, procena, display_labels=["laka", "teška"], cmap="Purples", colorbar=False
)
plt.title("Matrica konfuzije, prag 0,5")
plt.xlabel("Procena modela")
plt.ylabel("Stvarna klasa")
plt.show()

### Zadatak 3. Metrike iz matrice

Iz četiri broja izračunati tačnost, preciznost, odziv, specifičnost i F1, pa ih uporediti sa izveštajem `classification_report`.

In [ ]:
from sklearn.metrics import classification_report

tacnost = (tp + tn) / (tp + tn + fp + fn)
preciznost = tp / (tp + fp)
odziv = tp / (tp + fn)
specificnost = tn / (tn + fp)
f1 = 2 * preciznost * odziv / (preciznost + odziv)
print(f"tačnost      = {tacnost:.3f}")
print(f"preciznost   = {preciznost:.3f}")
print(f"odziv        = {odziv:.3f}")
print(f"specifičnost = {specificnost:.3f}")
print(f"F1           = {f1:.3f}")
print()
print(classification_report(y_te, procena, target_names=["laka", "teška"], digits=3))

## Prag odlučivanja

### Zadatak 4. Metrike za više pragova

Za pragove od 0,2 do 0,7 izračunati preciznost, odziv, F1 i broj obe vrste grešaka.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

redovi = []
for prag in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (p_te >= prag).astype(int)
    redovi.append(
        {
            "prag": prag,
            "preciznost": precision_score(y_te, p),
            "odziv": recall_score(y_te, p),
            "F1": f1_score(y_te, p),
            "propuštenih teških": int(((p == 0) & (y_te == 1)).sum()),
            "lažnih uzbuna": int(((p == 1) & (y_te == 0)).sum()),
        }
    )
print(pd.DataFrame(redovi).round(3).to_string(index=False))

## ROC i PR kriva

### Zadatak 5. ROC kriva i AUC

Nacrtati ROC krivu i izračunati AUC jednom iz verovatnoća, a jednom iz procena sa pragom 0,5.

AUC izračunat iz verovatnoća je 0,826, a iz procena 0,734. Druga vrednost je greška koja se često sreće u projektima: `roc_auc_score` se pozove sa rezultatom `predict` umesto `predict_proba`, pa se prijavi pogrešan broj bez ikakvog upozorenja.

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score

auc_verovatnoce = roc_auc_score(y_te, p_te)
auc_klase = roc_auc_score(y_te, procena)
print(f"ROC AUC iz verovatnoća:      {auc_verovatnoce:.3f}")
print(f"ROC AUC iz procena 0 ili 1:  {auc_klase:.3f}")

RocCurveDisplay.from_predictions(
    y_te, p_te, name="logistička regresija", color="#5b3cc4"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="nasumična procena")
plt.xlabel("Udeo lažnih uzbuna (FP / (FP + TN))")
plt.ylabel("Odziv (TP / (TP + FN))")
plt.title("ROC kriva")
plt.legend()
plt.show()

### Zadatak 6. PR kriva i prosečna preciznost

Nacrtati PR krivu, izračunati prosečnu preciznost i uporediti je sa udelom teških nezgoda.

AP je 0,771 naspram 0,406 za nasumičan model. Kod uravnoteženih klasa ROC i PR kriva govore slično. Razlika postaje velika kada je pozitivna klasa retka.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

ap = average_precision_score(y_te, p_te)
print(f"Prosečna preciznost (AP): {ap:.3f}")
print(f"Udeo teških (AP nasumičnog modela): {y_te.mean():.3f}")

PrecisionRecallDisplay.from_predictions(
    y_te, p_te, name="logistička regresija", color="#5b3cc4"
)
plt.axhline(y_te.mean(), linestyle="--", color="gray", label="nasumična procena")
plt.xlabel("Odziv")
plt.ylabel("Preciznost")
plt.title("PR kriva")
plt.legend()
plt.show()

## Retka klasa i cena grešaka

### Zadatak 7. Nezgode sa poginulim

Istrenirati isti model za procenu da li je nezgoda imala poginule, jednom sa podrazumevanim podešavanjem, a jednom sa `class_weight="balanced"`, i za oba izračunati tačnost, odziv, preciznost, ROC AUC i AP.

Nezgode sa poginulim čine 1,4 % testa, pa model koji ih nikad ne prijavi ima tačnost 0,986. Podrazumevani model upravo to i radi: odziv mu je nula. Sa težinama klasa odziv raste na 0,637, ali je preciznost samo 0,032: na svaku pronađenu nezgodu sa poginulim dolazi tridesetak lažnih uzbuna. ROC AUC oba modela je visok, 0,773 i 0,748, a AP nizak, 0,044 i 0,040, jedva iznad udela klase. Kod retke klase ROC AUC ulepšava sliku, a PR kriva pokazuje koliko je zadatak zaista težak. Težine klasa ne popravljaju rangiranje, AP čak i opada, nego pomeraju procene preko praga.

In [ ]:
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X, nezgode["poginuli"], test_size=0.2, stratify=nezgode["poginuli"], random_state=42
)
print(f"Udeo nezgoda sa poginulim u testu: {yp_te.mean():.4f}")
print(f"Tačnost modela koji nikad ne procenjuje poginule: {1 - yp_te.mean():.3f}\n")

for tezine in [None, "balanced"]:
    m = Pipeline(
        [
            ("priprema", priprema()),
            ("model", LogisticRegression(max_iter=1000, class_weight=tezine)),
        ]
    )
    m.fit(Xp_tr, yp_tr)
    p = m.predict_proba(Xp_te)[:, 1]
    k = (p >= 0.5).astype(int)
    print(
        f"class_weight = {str(tezine):8s}: tačnost = {(k == yp_te).mean():.3f}", end=""
    )
    print(f", odziv = {recall_score(yp_te, k):.3f}", end="")
    print(f", preciznost = {precision_score(yp_te, k, zero_division=0):.3f}")
    print(f"{'':25s}ROC AUC = {roc_auc_score(yp_te, p):.3f}", end="")
    print(f", AP = {average_precision_score(yp_te, p):.3f}")

### Zadatak 8. Prag prema ceni grešaka

Pretpostaviti da propuštena teška nezgoda košta pet puta više od lažne uzbune. Verovatnoće za trening skup dobiti unakrsnom validacijom, na njima izabrati prag sa najmanjom ukupnom cenom, pa uporediti prag 0,5 i izabrani prag na testu.

Izabrani prag je 0,15. Na testu propuštenih teških nezgoda ima 104 umesto 904, a ukupna cena pada sa 5.293 na 2.954. Preciznost pri tome pada na 0,515, što je svesno prihvaćena cena.

In [ ]:
from sklearn.model_selection import cross_val_predict

p_cv = cross_val_predict(model, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]

cena_fn, cena_fp = 5, 1
pragovi = np.round(np.arange(0.05, 0.96, 0.05), 2)
cene = []
for prag in pragovi:
    p = (p_cv >= prag).astype(int)
    fn_cv = ((p == 0) & (y_tr == 1)).sum()
    fp_cv = ((p == 1) & (y_tr == 0)).sum()
    cene.append(cena_fn * fn_cv + cena_fp * fp_cv)
najbolji_prag = pragovi[int(np.argmin(cene))]
print(f"Prag sa najmanjom cenom na unakrsnoj validaciji: {najbolji_prag}")

for prag in [0.5, najbolji_prag]:
    p = (p_te >= prag).astype(int)
    fn_te = int(((p == 0) & (y_te == 1)).sum())
    fp_te = int(((p == 1) & (y_te == 0)).sum())
    cena = cena_fn * fn_te + cena_fp * fp_te
    odziv_te, preciznost_te = recall_score(y_te, p), precision_score(y_te, p)
    print(f"test, prag {prag}: FN = {fn_te}, FP = {fp_te}, cena = {cena}", end="")
    print(f", odziv = {odziv_te:.3f}, preciznost = {preciznost_te:.3f}")

## Zadaci za samostalan rad

### Zadatak 9. Metrike ručno

Matrica konfuzije ima vrednosti TN = 50, FP = 10, FN = 5 i TP = 35. Ručno izračunati tačnost, preciznost, odziv i F1, pa rezultat proveriti kodom.

### Zadatak 10. Prag sa najvećim F1

Na verovatnoćama iz unakrsne validacije pronaći prag sa najvećim F1 i uporediti F1 na testu uz taj prag i uz prag 0,5.

### Zadatak 11. Prekoračenja PM10

Za model prekoračenja PM10 iz temperature i vetra sa vežbe 5 izračunati matricu konfuzije, odziv i preciznost, sa i bez težina klasa. Koliko dana sa prekoračenjem model propušta u svakom slučaju?

### Zadatak 12. kNN prema logističkoj regresiji

Za kNN sa 51 susedom izračunati ROC AUC i AP na testu i uporediti ih sa logističkom regresijom. Da li poredak modela zavisi od izabrane metrike?

### Zadatak 13. Najrizičnije nezgode

Poređati test nezgode po procenjenoj verovatnoći i izbrojati koliko je teških među prvih 100, 500 i 1000. Zašto ovakav način upotrebe modela ne zavisi od praga?